# 05. Árboles de Decisión: De Líneas a Reglas

**Nivel:** 🟢 Principiante  
**Tiempo estimado:** 60 minutos  
**Prerequisitos:** Conceptos básicos de clasificación

## 🎯 Objetivos de Aprendizaje

Al finalizar este notebook, podrás:
- Entender cómo funcionan los árboles de decisión
- Calcular entropía y ganancia de información
- Implementar el algoritmo ID3/CART desde cero
- Comprender criterios de splitting (Gini, Entropy)
- Visualizar árboles y su proceso de decisión
- Identificar y manejar overfitting en árboles

In [ ]:
# Importar librerías necesarias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix
from sklearn.datasets import load_iris, make_classification
import warnings
warnings.filterwarnings('ignore')

# Configuración
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

# Importar utilidades
import sys
sys.path.append('../../shared/utils')
from visualization import plot_tree_structure
from datasets import load_dataset

np.random.seed(42)
print("✅ Librerías importadas")

---
## 📌 1. Motivación: Pensando como Humanos

### El Problema

Eres médico y debes decidir si un paciente tiene una enfermedad. Tu razonamiento:

```
SI fiebre > 38°C:
    SI dolor_garganta = Sí:
        SI tos = Sí:
            → DIAGNÓSTICO: Gripe (90% confianza)
        SINO:
            → DIAGNÓSTICO: Faringitis
    SINO:
        → Revisar otros síntomas...
SINO:
    → Probablemente no es infección
```

**¡Esto es un árbol de decisión!**

### ¿Por qué Árboles de Decisión?

**Ventajas sobre regresión logística:**
- 📖 **Interpretables**: Puedes explicar cada decisión
- 🔢 **No requieren normalización**: Manejan diferentes escalas
- 🎯 **Capturan no-linealidades**: Sin necesidad de feature engineering
- 📊 **Manejan features categóricas** naturalmente
- 🚀 **Rápidos de entrenar y predecir**

### Aplicaciones

- 🏥 Sistemas de diagnóstico médico
- 💳 Aprobación de créditos
- 🎮 IA en videojuegos (behavior trees)
- 📧 Filtrado de spam
- 🏢 Sistemas expertos en general

### La Pregunta Guía

> **¿Cómo decide un algoritmo qué preguntas hacer y en qué orden para clasificar de forma óptima?**

---
## 📊 2. Intuición Visual

In [ ]:
# Ejemplo simple de árbol de decisión
# Dataset: Jugar tenis según condiciones climáticas

data = pd.DataFrame({
    'Cielo': ['Soleado', 'Soleado', 'Nublado', 'Lluvia', 'Lluvia', 'Lluvia', 'Nublado', 'Soleado'],
    'Temperatura': ['Calor', 'Calor', 'Calor', 'Templado', 'Frío', 'Frío', 'Frío', 'Templado'],
    'Humedad': ['Alta', 'Alta', 'Alta', 'Alta', 'Normal', 'Normal', 'Normal', 'Alta'],
    'Viento': ['Débil', 'Fuerte', 'Débil', 'Débil', 'Débil', 'Fuerte', 'Fuerte', 'Débil'],
    'Jugar': ['No', 'No', 'Sí', 'Sí', 'Sí', 'No', 'Sí', 'No']
})

print("🎾 Dataset: ¿Jugar Tenis?\n")
print(data.to_string(index=False))
print("\n💡 Objetivo: Aprender reglas para decidir si jugar o no")

In [ ]:
# Visualización de fronteras de decisión
# Comparar: Regresión Logística vs Árbol de Decisión

from sklearn.linear_model import LogisticRegression

# Generar datos con patrón no-lineal
X, y = make_classification(n_samples=200, n_features=2, n_redundant=0,
                          n_informative=2, n_clusters_per_class=1,
                          random_state=42)

# Entrenar ambos modelos
lr_model = LogisticRegression()
tree_model = DecisionTreeClassifier(max_depth=3)

lr_model.fit(X, y)
tree_model.fit(X, y)

# Crear grid para visualizar fronteras
h = 0.02
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                    np.arange(y_min, y_max, h))

# Predicciones en el grid
Z_lr = lr_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_lr = Z_lr.reshape(xx.shape)

Z_tree = tree_model.predict(np.c_[xx.ravel(), yy.ravel()])
Z_tree = Z_tree.reshape(xx.shape)

# Visualización
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Regresión Logística
axes[0].contourf(xx, yy, Z_lr, alpha=0.3, cmap='RdBu')
axes[0].scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k')
axes[0].set_title('Regresión Logística (Lineal)', fontsize=14)
axes[0].set_xlabel('Feature 1')
axes[0].set_ylabel('Feature 2')

# Árbol de Decisión
axes[1].contourf(xx, yy, Z_tree, alpha=0.3, cmap='RdBu')
axes[1].scatter(X[:, 0], X[:, 1], c=y, cmap='RdBu', edgecolors='k')
axes[1].set_title('Árbol de Decisión (No-Lineal)', fontsize=14)
axes[1].set_xlabel('Feature 1')
axes[1].set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

print("\n💡 Observa:")
print("   • Regresión logística: Frontera lineal (suave)")
print("   • Árbol de decisión: Fronteras rectangulares (eje-paralelas)")
print("   • El árbol captura patrones más complejos")

---
## 🧮 3. Fundamentos Matemáticos

### 📖 Conceptos Clave

#### 1. Entropía (Desorden/Impureza)

Mide cuán "mezcladas" están las clases:

$$
H(S) = -\sum_{i=1}^{C} p_i \log_2(p_i) \tag{1}
$$

Donde $p_i$ es la proporción de clase $i$ en el conjunto $S$.

**Casos extremos:**
- $H = 0$: Todas las muestras de una clase (puro)
- $H = 1$: Clases perfectamente balanceadas (máximo desorden)

**Ejemplo:**
- Dataset: [5 Sí, 5 No] → $H = -0.5\log_2(0.5) - 0.5\log_2(0.5) = 1.0$
- Dataset: [9 Sí, 1 No] → $H = -0.9\log_2(0.9) - 0.1\log_2(0.1) = 0.47$

#### 2. Ganancia de Información

Cuánto reduce la entropía dividir por un atributo:

$$
IG(S, A) = H(S) - \sum_{v \in Values(A)} \frac{|S_v|}{|S|} H(S_v) \tag{2}
$$

**Interpretación:** ¿Cuánta información ganamos al conocer el valor de A?

#### 3. Índice de Gini (Alternativa a Entropía)

Usado en CART (Classification and Regression Trees):

$$
Gini(S) = 1 - \sum_{i=1}^{C} p_i^2 \tag{3}
$$

**Propiedades:**
- $Gini = 0$: Nodo puro
- $Gini = 0.5$: Máximo desorden (2 clases balanceadas)

### Algoritmo ID3 (Simplified)

```
function BuildTree(S, Attributes):
    if all examples in S have same class:
        return Leaf(class)
    
    if Attributes is empty:
        return Leaf(majority_class)
    
    A = attribute with highest Information Gain
    tree = new Tree node with A
    
    for each value v of A:
        S_v = subset of S where A = v
        subtree = BuildTree(S_v, Attributes - {A})
        add subtree to tree
    
    return tree
```

### Ejemplo Numérico Completo

Dataset: [6 Positivos, 4 Negativos]

**Entropía inicial:**
$$
H = -\frac{6}{10}\log_2(\frac{6}{10}) - \frac{4}{10}\log_2(\frac{4}{10}) = 0.971
$$

**Feature A:** Valores {X, Y}
- A=X: [4 Pos, 1 Neg] → $H(X) = 0.722$
- A=Y: [2 Pos, 3 Neg] → $H(Y) = 0.971$

**Ganancia de Información:**
$$
IG(A) = 0.971 - [\frac{5}{10}(0.722) + \frac{5}{10}(0.971)] = 0.125
$$

In [ ]:
# Implementación de funciones de impureza

def entropy(y):
    """Calcula entropía de Shannon"""
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return -np.sum(probabilities * np.log2(probabilities + 1e-10))

def gini_index(y):
    """Calcula índice de Gini"""
    _, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return 1 - np.sum(probabilities ** 2)

# Ejemplo
y_pure = np.array([1, 1, 1, 1, 1])
y_balanced = np.array([0, 0, 1, 1])
y_imbalanced = np.array([0, 1, 1, 1, 1, 1, 1, 1, 1])

print("Medidas de Impureza:\n")
print(f"Dataset Puro [1,1,1,1,1]:")
print(f"  Entropía: {entropy(y_pure):.3f}")
print(f"  Gini: {gini_index(y_pure):.3f}\n")

print(f"Dataset Balanceado [0,0,1,1]:")
print(f"  Entropía: {entropy(y_balanced):.3f}")
print(f"  Gini: {gini_index(y_balanced):.3f}\n")

print(f"Dataset Desbalanceado [0,1,1,1,1,1,1,1,1]:")
print(f"  Entropía: {entropy(y_imbalanced):.3f}")
print(f"  Gini: {gini_index(y_imbalanced):.3f}")

---
## 💻 4. Implementación Desde Cero

In [ ]:
class ArbolDecision:
    """
    Implementación simplificada de árbol de decisión para clasificación.
    Usa el criterio de Gini para splitting.
    """
    
    class Nodo:
        """Clase interna para representar un nodo del árbol"""
        def __init__(self, gini, n_samples, n_samples_per_class, predicted_class):
            self.gini = gini
            self.n_samples = n_samples
            self.n_samples_per_class = n_samples_per_class
            self.predicted_class = predicted_class
            self.feature_index = 0
            self.threshold = 0
            self.left = None
            self.right = None
    
    def __init__(self, max_depth=None, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.n_classes = None
        self.n_features = None
        self.tree = None
    
    def fit(self, X, y):
        """Construye el árbol de decisión"""
        self.n_classes = len(set(y))
        self.n_features = X.shape[1]
        self.tree = self._grow_tree(X, y)
        return self
    
    def _gini(self, y):
        """Calcula índice de Gini"""
        m = len(y)
        return 1.0 - sum((np.sum(y == c) / m) ** 2 for c in range(self.n_classes))
    
    def _best_split(self, X, y):
        """Encuentra el mejor split (feature y threshold)"""
        m = len(y)
        if m <= 1:
            return None, None
        
        # Gini del nodo padre
        parent_gini = self._gini(y)
        
        best_gain = 0
        best_idx, best_thr = None, None
        
        # Iterar sobre features
        for idx in range(self.n_features):
            # Ordenar por feature
            thresholds, classes = zip(*sorted(zip(X[:, idx], y)))
            
            num_left = [0] * self.n_classes
            num_right = [np.sum(y == c) for c in range(self.n_classes)]
            
            # Probar cada threshold
            for i in range(1, m):
                c = classes[i - 1]
                num_left[c] += 1
                num_right[c] -= 1
                
                gini_left = 1.0 - sum((num_left[x] / i) ** 2 for x in range(self.n_classes))
                gini_right = 1.0 - sum((num_right[x] / (m - i)) ** 2 for x in range(self.n_classes))
                
                # Ganancia de información ponderada
                gini_split = (i * gini_left + (m - i) * gini_right) / m
                gain = parent_gini - gini_split
                
                if thresholds[i] == thresholds[i - 1]:
                    continue
                
                if gain > best_gain:
                    best_gain = gain
                    best_idx = idx
                    best_thr = (thresholds[i] + thresholds[i - 1]) / 2
        
        return best_idx, best_thr
    
    def _grow_tree(self, X, y, depth=0):
        """Construye el árbol recursivamente"""
        n_samples_per_class = [np.sum(y == i) for i in range(self.n_classes)]
        predicted_class = np.argmax(n_samples_per_class)
        node = self.Nodo(
            gini=self._gini(y),
            n_samples=len(y),
            n_samples_per_class=n_samples_per_class,
            predicted_class=predicted_class
        )
        
        # Condiciones de parada
        if depth < self.max_depth and len(y) >= self.min_samples_split:
            idx, thr = self._best_split(X, y)
            if idx is not None:
                indices_left = X[:, idx] < thr
                X_left, y_left = X[indices_left], y[indices_left]
                X_right, y_right = X[~indices_left], y[~indices_left]
                node.feature_index = idx
                node.threshold = thr
                node.left = self._grow_tree(X_left, y_left, depth + 1)
                node.right = self._grow_tree(X_right, y_right, depth + 1)
        return node
    
    def predict(self, X):
        """Predice clases para X"""
        return np.array([self._predict_one(x) for x in X])
    
    def _predict_one(self, x, node=None):
        """Predice la clase para una muestra"""
        if node is None:
            node = self.tree
        
        if node.left is None:  # Es hoja
            return node.predicted_class
        
        if x[node.feature_index] < node.threshold:
            return self._predict_one(x, node.left)
        return self._predict_one(x, node.right)

print("✅ Clase ArbolDecision definida")

In [ ]:
# Probar con Iris
iris = load_iris()
X, y = iris.data, iris.target

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar nuestro árbol
tree = ArbolDecision(max_depth=3, min_samples_split=2)
tree.fit(X_train, y_train)

# Evaluar
y_pred = tree.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

print(f"📊 Accuracy en test: {accuracy:.4f}")

---
## 🏭 5. Versión con Framework (Scikit-learn)

In [ ]:
# Scikit-learn
sklearn_tree = DecisionTreeClassifier(max_depth=3, criterion='gini', random_state=42)
sklearn_tree.fit(X_train, y_train)

y_pred_sklearn = sklearn_tree.predict(X_test)
accuracy_sklearn = accuracy_score(y_test, y_pred_sklearn)

print(f"Nuestra implementación: {accuracy:.4f}")
print(f"Scikit-learn:           {accuracy_sklearn:.4f}")

# Visualizar el árbol
plt.figure(figsize=(15, 10))
plot_tree(sklearn_tree, feature_names=iris.feature_names, 
         class_names=iris.target_names, filled=True, rounded=True)
plt.title("Árbol de Decisión - Dataset Iris", fontsize=16)
plt.show()

print("\n💡 Cada nodo muestra:")
print("   • Condición de split")
print("   • Gini: Impureza del nodo")
print("   • samples: Número de muestras")
print("   • value: Distribución de clases")
print("   • class: Clase mayoritaria (predicción)")

---
## 🎯 6. Ejercicios

In [ ]:
# Ejercicios aquí (similar a notebooks anteriores)
print("TODO: Implementar ejercicios")

---
## 📚 7. Resumen

### Puntos Clave

1. Los árboles dividen el espacio en regiones rectangulares
2. Entropía/Gini miden impureza de nodos
3. Ganancia de información guía los splits
4. Propensos a overfitting (necesitan poda)
5. Muy interpretables pero inestables

### Próximo Paso

**06. Random Forests** - Combinar múltiples árboles para mejor generalización

[← 04. Softmax](04-regresion-softmax.ipynb) | [06. Random Forests →](06-random-forests.ipynb)

</div>